# Spam model selection

Candidates are compared only on the labeled Averis dataset. The GitHub reference's TF-IDF + MLP is included alongside count, word, character, and combined word/character approaches.

Protocol:

1. Build the exact text representation used by production inference.
2. Remove exact duplicate text before splitting.
3. Reserve a fixed 20% stratified holdout and never use it for selection.
4. Rank candidates using repeated stratified cross-validation on the training partition.
5. Tune the winner's threshold from out-of-fold training predictions.
6. Evaluate the selected pipeline once on the holdout.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.naive_bayes import BernoulliNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)
RANDOM_STATE = 42

In [2]:
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
DATA_ROOT = ROOT / "docs-provided/problem-statement/sdoc-hackathon-docker/data_v2"
labels = json.loads((DATA_ROOT / "ground_truth.json").read_text())

records = {}
for path in sorted((DATA_ROOT / "inbox").glob("*.json")):
    email = json.loads(path.read_text())
    text = f"From: {email.get('from', '')}\nSubject: {email.get('subject', '')}\nBody: {email.get('body', '')}"
    label = int(labels[email["email_id"]]["category"] == "SPAM")
    if text in records:
        assert records[text] == label
    records[text] = label

X = np.array(list(records), dtype=object)
y = np.array(list(records.values()))
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)
split_summary = {
    "after_deduplication": len(X),
    "train": len(X_train),
    "train_spam": int(y_train.sum()),
    "test": len(X_test),
    "test_spam": int(y_test.sum()),
}
print(json.dumps(split_summary, indent=2))

{
  "after_deduplication": 519,
  "train": 415,
  "train_spam": 31,
  "test": 104,
  "test_spam": 8
}


In [3]:
def word_tfidf():
    return TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        strip_accents="unicode",
    )


def char_tfidf():
    return TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        sublinear_tf=True,
        max_features=30_000,
    )


candidates = {
    "majority_baseline": Pipeline(
        [("vectorizer", word_tfidf()), ("classifier", DummyClassifier(strategy="prior"))]
    ),
    "reference_word_tfidf_mlp": Pipeline(
        [
            (
                "vectorizer",
                TfidfVectorizer(
                    binary=True,
                    max_df=0.9,
                    min_df=0.001,
                    stop_words="english",
                    sublinear_tf=True,
                ),
            ),
            (
                "classifier",
                MLPClassifier(
                    alpha=0.1,
                    hidden_layer_sizes=(20, 30, 20),
                    max_iter=4_000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "count_bernoulli_nb": Pipeline(
        [
            (
                "vectorizer",
                CountVectorizer(
                    binary=True,
                    ngram_range=(1, 2),
                    min_df=2,
                    stop_words="english",
                ),
            ),
            ("classifier", BernoulliNB(alpha=0.1)),
        ]
    ),
    "count_random_forest": Pipeline(
        [
            ("vectorizer", CountVectorizer(binary=True, ngram_range=(1, 2), min_df=2)),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=300,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]
    ),
    "word_tfidf_logistic": Pipeline(
        [
            ("vectorizer", word_tfidf()),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1_000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "char_tfidf_logistic": Pipeline(
        [
            ("vectorizer", char_tfidf()),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1_000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "word_char_tfidf_logistic": Pipeline(
        [
            ("vectorizer", FeatureUnion([("word", word_tfidf()), ("char", char_tfidf())])),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1_000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "word_tfidf_calibrated_svc": Pipeline(
        [
            ("vectorizer", word_tfidf()),
            (
                "classifier",
                CalibratedClassifierCV(
                    LinearSVC(class_weight="balanced", random_state=RANDOM_STATE),
                    method="sigmoid",
                    cv=3,
                ),
            ),
        ]
    ),
}
list(candidates)

['majority_baseline',
 'reference_word_tfidf_mlp',
 'count_bernoulli_nb',
 'count_random_forest',
 'word_tfidf_logistic',
 'char_tfidf_logistic',
 'word_char_tfidf_logistic',
 'word_tfidf_calibrated_svc']

In [4]:
def false_positive_rate_scorer(estimator, features, target):
    tn, fp, _, _ = confusion_matrix(target, estimator.predict(features), labels=[0, 1]).ravel()
    return fp / (fp + tn) if fp + tn else 0.0


cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
scoring = {
    "spam_f1": make_scorer(f1_score, zero_division=0),
    "spam_precision": make_scorer(precision_score, zero_division=0),
    "spam_recall": make_scorer(recall_score),
    "pr_auc": "average_precision",
    "false_positive_rate": false_positive_rate_scorer,
}
results = []
for name, candidate in candidates.items():
    scores = cross_validate(candidate, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results.append(
        {
            "candidate": name,
            "spam_f1_mean": float(scores["test_spam_f1"].mean()),
            "spam_f1_std": float(scores["test_spam_f1"].std()),
            "spam_precision_mean": float(scores["test_spam_precision"].mean()),
            "spam_recall_mean": float(scores["test_spam_recall"].mean()),
            "pr_auc_mean": float(scores["test_pr_auc"].mean()),
            "false_positive_rate_mean": float(scores["test_false_positive_rate"].mean()),
            "fit_seconds_mean": float(scores["fit_time"].mean()),
        }
    )

ranking = sorted(
    results,
    key=lambda row: (
        row["spam_f1_mean"],
        row["pr_auc_mean"],
        -row["false_positive_rate_mean"],
        -row["fit_seconds_mean"],
    ),
    reverse=True,
)
print(json.dumps(ranking, indent=2))

[
  {
    "candidate": "word_tfidf_logistic",
    "spam_f1_mean": 1.0,
    "spam_f1_std": 0.0,
    "spam_precision_mean": 1.0,
    "spam_recall_mean": 1.0,
    "pr_auc_mean": 1.0,
    "false_positive_rate_mean": 0.0,
    "fit_seconds_mean": 0.07772879600524903
  },
  {
    "candidate": "word_tfidf_calibrated_svc",
    "spam_f1_mean": 1.0,
    "spam_f1_std": 0.0,
    "spam_precision_mean": 1.0,
    "spam_recall_mean": 1.0,
    "pr_auc_mean": 1.0,
    "false_positive_rate_mean": 0.0,
    "fit_seconds_mean": 0.114729372660319
  },
  {
    "candidate": "char_tfidf_logistic",
    "spam_f1_mean": 1.0,
    "spam_f1_std": 0.0,
    "spam_precision_mean": 1.0,
    "spam_recall_mean": 1.0,
    "pr_auc_mean": 1.0,
    "false_positive_rate_mean": 0.0,
    "fit_seconds_mean": 0.4374138355255127
  },
  {
    "candidate": "word_char_tfidf_logistic",
    "spam_f1_mean": 1.0,
    "spam_f1_std": 0.0,
    "spam_precision_mean": 1.0,
    "spam_recall_mean": 1.0,
    "pr_auc_mean": 1.0,
    "false_positive_

In [5]:
winner_name = ranking[0]["candidate"]
winner = candidates[winner_name]
selection_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_scores = cross_val_predict(
    winner,
    X_train,
    y_train,
    cv=selection_cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

threshold_results = []
for threshold in np.linspace(0.05, 0.95, 181):
    predicted = (oof_scores >= threshold).astype(int)
    tn, fp, _, _ = confusion_matrix(y_train, predicted, labels=[0, 1]).ravel()
    threshold_results.append(
        {
            "threshold": float(threshold),
            "spam_f1": float(f1_score(y_train, predicted)),
            "spam_precision": float(precision_score(y_train, predicted, zero_division=0)),
            "spam_recall": float(recall_score(y_train, predicted)),
            "false_positive_rate": float(fp / (fp + tn)),
        }
    )

best_threshold = max(
    threshold_results,
    key=lambda row: (row["spam_f1"], -row["false_positive_rate"], row["spam_precision"]),
)
winner.fit(X_train, y_train)
test_scores = winner.predict_proba(X_test)[:, 1]
test_predictions = (test_scores >= best_threshold["threshold"]).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_predictions, labels=[0, 1]).ravel()
holdout = {
    "spam_f1": float(f1_score(y_test, test_predictions)),
    "spam_precision": float(precision_score(y_test, test_predictions, zero_division=0)),
    "spam_recall": float(recall_score(y_test, test_predictions)),
    "pr_auc": float(average_precision_score(y_test, test_scores)),
    "false_positive_rate": float(fp / (fp + tn)),
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
}
selection = {
    "winner": winner_name,
    "threshold_from_training_oof": best_threshold,
    "holdout": holdout,
}
print(json.dumps(selection, indent=2))

{
  "winner": "word_tfidf_logistic",
  "threshold_from_training_oof": {
    "threshold": 0.25499999999999995,
    "spam_f1": 1.0,
    "spam_precision": 1.0,
    "spam_recall": 1.0,
    "false_positive_rate": 0.0
  },
  "holdout": {
    "spam_f1": 1.0,
    "spam_precision": 1.0,
    "spam_recall": 1.0,
    "pr_auc": 1.0,
    "false_positive_rate": 0.0,
    "confusion_matrix": {
      "tn": 96,
      "fp": 0,
      "fn": 0,
      "tp": 8
    }
  }
}


## Interpretation guardrail

The winner is selected by repeated training-fold spam F1, with PR-AUC as the tie-breaker. The threshold is derived only from out-of-fold predictions on the training partition. The holdout contains approximately eight spam messages, so even perfect holdout metrics are weak evidence and describe only this synthetic generator. Production monitoring and independently labeled real email are required before treating the model as generally reliable.